In [1]:
%pip install tqdm
%pip install joblib
%pip install tensorflow
%pip install -U hats lsdb
%pip install "dask[complete]"
%pip install 'light-curve[full]'

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import time
import numpy as np
import pandas as pd
from tqdm import tqdm
import tensorflow as tf
from lsdb import read_hats
from dask.distributed import Client
from joblib import Parallel, delayed
from nested_pandas import NestedDtype
from joblib import wrap_non_picklable_objects


In [171]:
def _bytes_feature(value):
    """Returns a bytes_list from a string / byte."""
    if isinstance(value, type(tf.constant(0))):
        value = value.numpy() # BytesList won't unpack a string from an EagerTensor.
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def _float_feature(list_of_floats):  # float32
    return tf.train.Feature(float_list=tf.train.FloatList(value=list_of_floats))

def _int64_feature(value):
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[value]))

def get_example(lcid, label, lightcurve, meta):
    """
    Create a record example from numpy values.

    Args:
        lcid (string): object id
        label (int): class code
        lightcurve (numpy array): time, magnitudes and observational error

    Returns:
        tensorflow record
    """

    f = dict()

    dict_features={
    'id': tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(lcid).encode()])),
    'label': tf.train.Feature(int64_list=tf.train.Int64List(value=[label])),
    'length': tf.train.Feature(int64_list=tf.train.Int64List(value=[lightcurve.shape[0]])),
    'meta': _float_feature(meta)
    }
    element_context = tf.train.Features(feature = dict_features)

    dict_sequence = {}
    for col in range(lightcurve.shape[1]):
        seqfeat = _float_feature(lightcurve[:, col])
        seqfeat = tf.train.FeatureList(feature = [seqfeat])
        dict_sequence['dim_{}'.format(col)] = seqfeat

    element_lists = tf.train.FeatureLists(feature_list=dict_sequence)
    ex = tf.train.SequenceExample(context = element_context,
                                  feature_lists= element_lists)
    return ex


def train_test_split(frame, train):
    """
    Divide the dataset into train, validation and test subsets.
    Notice that:
        test = 1 - (train + val)

    Args:
        frame (Dataframe): Dataframe following the astro-standard format
        dest (string): Record destination.
        train (float): train fraction
        val (float): validation fraction
    Returns:
        tuple x3 : (name of subset, subframe with metadata)
    """

    frame = frame.sample(frac=1)
    n_samples = frame.shape[0]

    n_train = int(n_samples*train)
   

    sub_test  = frame.iloc[:]
    sub_train = frame.iloc[:n_train]
    sub_val   = frame.iloc[n_train:]
    
    return ('train', sub_train), ('val', sub_val), ('test', sub_test)

def process_lc3(lc_index, label, numpy_lc, numpy_meta, writer):
    try:
        ex = get_example(lc_index, label, numpy_lc, numpy_meta)
        writer.write(ex.SerializeToString())
    except Exception as e:
        print('[INFO] {} could not be processed'.format(lc_index))


@wrap_non_picklable_objects
def process_lc2(row, source, meta, unique_classes, **kwargs):
    path  = row['Path'].split('/')[-1]
    label = list(unique_classes).index(row['Class'])
    lc_path = os.path.join(source, path)
    meta_path = os.path.join(meta, path)
    observations = pd.read_csv(lc_path, **kwargs)
    meta_data = pd.read_csv(meta_path, **kwargs)
    observations.columns = ["mjd", "mag", "magerr"]
    observations = observations.dropna()
    # observations.sort_values('days')
    # observations = observations.drop_duplicates(keep='last')
   

    numpy_lc = observations.values
    numpy_meta = meta_data.values

    return row['ID'], label, numpy_lc, numpy_meta[0]


def write_records(frame, dest, max_lcs_per_record, source, meta, unique, n_jobs=None, max_obs=200, **kwargs):
    # Get frames with fixed number of lightcurves
    collection = [frame.iloc[i:i+max_lcs_per_record] \
                  for i in range(0, frame.shape[0], max_lcs_per_record)]

    for counter, subframe in enumerate(collection):
        var = Parallel(n_jobs=n_jobs)(delayed(process_lc2)(row, source, meta, unique, **kwargs) \
                                    for k, row in subframe.iterrows())

        with tf.io.TFRecordWriter(dest+'/chunk_{}.record'.format(counter)) as writer:
            for counter2, data_lc in enumerate(var):
                process_lc3(*data_lc, writer)





In [167]:
def create_dataset(df,
                    target,
                    train_size=0.80,
                    max_lcs_per_chunk=100):

    if not train_size or train_size > 1:
        raise AssertionError(f"Please provide a valid train_size fraction. Got {train_size}")
    
    if not os.path.exists(target):
        os.makedirs(target, exist_ok=True)
    
    info_df = pd.DataFrame()
    bands = {'g': np.log10(4835.99), 'r': np.log10(6468.25), 'i': np.log10(7993.65)}
    last_index = dict()
    LC_COLUMN = "lc"
    
    df = df.assign(**{LC_COLUMN: df[LC_COLUMN].astype(NestedDtype.from_pandas_arrow_dtype(df.dtypes[LC_COLUMN]))},)
    df = df.dropna(subset=['lc'])
    df = df["lc"]  
    for id in df.index:
        temp_df = pd.DataFrame(df[id])
        mask_id = (temp_df["info"] == 0) & (temp_df["flag"] == 0)
        temp_df = temp_df[mask_id]
        temp_df['band_sorted'] = pd.Categorical( temp_df['band'], 
                                        categories=list(bands.keys()), 
                                        ordered=True
                                        )
        temp_df = temp_df.sort_values('band_sorted')
        temp_df = temp_df.reset_index(drop=True)
        temp_df.drop(columns=['band'], inplace=True)

        
        for k, v in bands.items():
                if k == 'g':
       
                        end_idx = temp_df[temp_df["band_sorted"] == k].shape[0]-1
                        last_index[k] = end_idx
                else:
        
                        end_idx = temp_df[temp_df["band_sorted"] == k].shape[0] + end_idx
                        last_index[k] = end_idx
    
    return temp_df
    

    unique, counts = np.unique(df['Class'], return_counts=True)
    info_df['label'] = unique
    info_df['size'] = counts
    # info_df.to_csv(os.path.join(target, 'objects.csv'), index=False)
    

    # Separate by class
    cls_groups = df.groupby('Class')
    
    for cls_name, cls_meta in tqdm(cls_groups, total=len(cls_groups)):
        subsets = train_test_split(cls_meta, train=train_size)

        for subset_name, frame in subsets:
            if frame is None:
                continue
            dest = os.path.join(target, subset_name, cls_name)
            os.makedirs(dest, exist_ok=True)
            write_records(frame, dest, max_lcs_per_chunk, source, meta, unique, n_jobs, **kwargs)


    

In [168]:
%%time
# Read the HATS dataset
read_catalog = read_hats('./cephids/zubercal_vcep', )
# print(read_catalog.head())
# with Client() as client:
#     catalog =  read_catalog.compute()

# print("\nCatalog read!", type(catalog))

# import dask.dataframe as dd

# catalog = dd.from_pandas(catalog)
    
print("\nStarting!")
start = time.time()
#
#
#
path_to_buff="./cephids/zubercal_vcep"
path_to_store="./cephids/zubercal_vcep"
#
#
#
processed_df = read_catalog._ddf.map_partitions(create_dataset, target=path_to_store,
                                                        train_size=0.80,
                                                        max_lcs_per_chunk=100)
df = processed_df.compute()
df

# with Client() as client:
#     precessed_df.compute()

# end = time.time()
# print("\nTime (mins):", ((end-start)//60))
# print("\nDone!")

    


Starting!


ValueError: Metadata inference failed in `create_dataset`.

You have supplied a custom function and Dask is unable to 
determine the type of output that that function returns. 

To resolve this please provide a meta= keyword.
The docstring of the Dask function you ran should have more information.

Original error is below:
------------------------
UnboundLocalError("cannot access local variable 'temp_df' where it is not associated with a value")

Traceback:
---------
  File "/Users/torshamajumder/Pycharm_Projects/test/.venv/lib/python3.12/site-packages/dask/dataframe/utils.py", line 133, in raise_on_meta_error
    yield
  File "/Users/torshamajumder/Pycharm_Projects/test/.venv/lib/python3.12/site-packages/dask/dataframe/dask_expr/_expr.py", line 4042, in emulate
    return func(*_extract_meta(args, True), **_extract_meta(kwargs, True))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/cw/bz7q_b9d0rz4j8xvw9_9mds40000gn/T/ipykernel_10956/3485

In [170]:
bands = {'g': np.log10(4835.99), 'r': np.log10(6468.25), 'i': np.log10(7993.65)}
last_index = dict()
for id in df.index:
        temp_df = pd.DataFrame(df[id])
        mask_id = (temp_df["info"] == 0) & (temp_df["flag"] == 0)
        temp_df = temp_df[mask_id]
        temp_df['band_sorted'] = pd.Categorical( temp_df['band'], 
                                        categories=list(bands.keys()), 
                                        ordered=True
                                        )
        temp_df = temp_df.sort_values('band_sorted')
        temp_df = temp_df.reset_index(drop=True)
        temp_df.drop(columns=['band'], inplace=True)

        
        for k, v in bands.items():
                if k == 'g':
       
                        end_idx = temp_df[temp_df["band_sorted"] == k].shape[0]-1
                        last_index[k] = end_idx
                else:
        
                        end_idx = temp_df[temp_df["band_sorted"] == k].shape[0] + end_idx
                        last_index[k] = end_idx
        



print("Sorted df based on g, r, i bands --> \n",temp_df.head())
print("Last index of the g, r, i bands in the df --> \n", last_index) 
print("Log value of the band filters --> \n", bands)


Sorted df based on g, r, i bands --> 
             mjd        mag  magerr  info  flag band_sorted
0  59531.168168  16.317699     489     0     0           g
1  59378.423946  16.205000     104     0     0           g
2  59362.421650  16.070999     117     0     0           g
3  58702.336483  16.215000     122     0     0           g
4  58712.213360  15.622600     100     0     0           g
Last index of the g, r, i bands in the df --> 
 {'g': 232, 'r': 797, 'i': 818}
Log value of the band filters --> 
 {'g': np.float64(3.6844853941429716), 'r': np.float64(3.8107867971839995), 'i': np.float64(3.9027451288632498)}
